In [13]:
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
from torchvision.transforms import Normalize
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from kaggle_secrets import UserSecretsClient
import os
import json
import shutil
import subprocess

user_secrets = UserSecretsClient()
kaggle_username = user_secrets.get_secret("KAGGLE_USERNAME")
dataset_dir = "/kaggle/working/dataset_batch"

# clear working dir
print("clearing working directory")
for item in os.listdir('/kaggle/working'):
    item_path = os.path.join('/kaggle/working', item)
    if os.path.isfile(item_path):
        os.remove(item_path)
    elif os.path.isdir(item_path) and item != '.virtual_documents':
        shutil.rmtree(item_path)

os.makedirs(dataset_dir, exist_ok=True)
print("working directory cleared")

torch.hub._validate_not_a_forked_repo = lambda a, b, c: True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"using device {device}")


# encoders
class DinoEncoder(nn.Module):
    def __init__(self, base_model, feature_key):
        super().__init__()
        self.base_model = base_model
        self.feature_key = feature_key
        self.emb_dim = base_model.num_features
        self.patch_size = base_model.patch_size
        if feature_key == "x_norm_patchtokens":
            self.latent_ndim = 2
        elif feature_key == "x_norm_clstoken":
            self.latent_ndim = 1
        else:
            raise ValueError(f"invalid feature key {feature_key}")

    def forward(self, x):
        emb = self.base_model.forward_features(x)[self.feature_key]
        if self.latent_ndim == 1:
            emb = emb.unsqueeze(1)
        return emb


class DinoV2Encoder(DinoEncoder):
    def __init__(self, name, feature_key):
        base_model = torch.hub.load(
            "facebookresearch/dinov2", 
            name
        )
        super().__init__(base_model, feature_key)
        self.name = name


class DinoV3Encoder(DinoEncoder):
    # DinoV3 allows for the option to locally add weights, set local=True and
    # input the dir path of preferred weights
    def __init__(self, name, feature_key, local=False, weights=None):
        if local:
            base_model = torch.hub.load(
                "facebookresearch/dinov3",
                name,
                source='local',
                weights=weights,
            )
        else:
            base_model = torch.hub.load(
                "facebookresearch/dinov3",
                name,
                pretrained=False
            )
        super().__init__(base_model, feature_key)
        self.name = name


# encoder function to pre-compute embeddings in batches
def encode(batch_size, batch_no, src_path, dataset_dir, feature_key, encoder_ver, 
           model_ver="DinoV2", float_size="32"):
    encoder = None
    if model_ver == "DinoV2":
        encoder = DinoV2Encoder(encoder_ver, feature_key).to(device)
    else:
        encoder = DinoV3Encoder(encoder_ver, feature_key).to(device)
    encoder.eval()

    dataset_name = f"{model_ver}-pointmaze-embeddings"
    dataset_id = f"{kaggle_username}/{dataset_name}"

    normalization_vector = Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    
    with torch.no_grad():
        start_index = (batch_no - 1) * batch_size
        end_index = batch_no * batch_size
        for i in range(start_index, end_index):
            filename = f"episode_{i:03d}.pth"
    
            tensor = torch.load(f"{src_path}{filename}", map_location='cpu')
            tensor = tensor.float() / 255.0
            # permute the tensor s.t. it adheres to right tensor format required by Dinov2
            tensor = tensor.permute(0, 3, 1, 2)
            tensor = normalization_vector(tensor).to(device)

            if float_size == "16":
                tensor = tensor.half()
            feature = encoder(tensor).cpu()
    
            save_filename = f"{dataset_dir}/patched_ep{i:03d}.pt"
            torch.save(feature, save_filename)
    
            if i % 10 == 0:
                print(f"saved patched tensor {i:03d}")

            # save embedded tensors to datasets in batches to bypass Kaggle's
            # 19.5GiB working directory limit
            if (i + 1) % end_index == 0:
                print(f"uploading batch up to episode {i:03d}")

                # make sure that a dataset with the same name does not exist yet
                # (i.e. dataset id should be unique)
                dataset_metadata = {
                    "title": f"{model_ver} PointMaze Embeddings-part{batch_no}",
                    "id": dataset_id,
                    "licenses": [{"name": "CC0-1.0"}],
                }
                with open(f"{dataset_dir}/dataset-metadata.json", 'w') as f:
                    json.dump(dataset_metadata, f)
    
                try:
                    create_dataset_cmd = f"kaggle datasets create -p {dataset_dir} --dir-mode zip"
                    subprocess.run(create_dataset_cmd, shell=True, check=True)
                    print(f"uploaded batch {i:03d} clearing local files")
    
                except subprocess.CalledProcessError as e:
                    print(f"error uploading to kaggle {e}")
                    break
    
                for fname in os.listdir(dataset_dir):
                    if fname.endswith('.pt'):
                        os.remove(os.path.join(dataset_dir, fname))

clearing working directory
working directory cleared
using device cuda


In [16]:
# set output path and other config params (replace src_path with dataset of choice)
src_path = "/kaggle/input/datasets/thegreenier/pointmaze/point_maze/obses/"
feature_key = "x_norm_patchtokens"

# DinoV2 choices: dinov2_vits14, dinov2_vitb14, dinov2_vitl14, dinov2_vitg14
# https://github.com/facebookresearch/dinov2
# DinoV3 choices: dinov3_vits16, dinov3_vitb16, dinov3_vitl16, dinov3_vitg16
# https://github.com/facebookresearch/dinov3
# In increasing order of parameters.
encoder_ver = "dinov3_vits16"
batch_size = 500

batch_no = 1
encode(batch_size, batch_no, src_path, dataset_dir, feature_key, encoder_ver, model_ver="DinoV3")

In [ ]:
batch_no = 2
encode(batch_size, batch_no, src_path, dataset_dir, feature_key, encoder_ver, model_ver="DinoV3")

In [ ]:
batch_no = 3
encode(batch_size, batch_no, src_path, dataset_dir, feature_key, encoder_ver, model_ver="DinoV3")

In [ ]:
batch_no = 4
encode(batch_size, batch_no, src_path, dataset_dir, feature_key, encoder_ver, model_ver="DinoV3")